# 03 — Metrics + Report

Sau khi inference xong (NB2), compute metrics + generate reports.

**Speedups on Colab T4 vs RTX 3050 Laptop**:
- BGE-M3 embed (answer_relevance): ~3× faster
- BERTScore (BERT multilingual, n=1000): ~5× faster
- compute_metrics overall: ~2× faster

Pipeline:
1. Re-setup env
2. Sync data from GDrive to /content (faster IO)
3. (Optional) Backfill plain_answer
4. Apply audit fixes (pairwise + split halluc + tag api_err)
5. Compute metrics (R1 + R2)
6. Generate reports + significance tests
7. Sync output → GDrive
8. (Optional) Download reports zip

## 3.1 Setup environment

In [ ]:
from google.colab import drive, userdata
import os, sys
drive.mount('/content/drive')
%cd /content/legal-graph-kb
sys.path.insert(0, '/content/legal-graph-kb')

for k in ['OPENAI_API_KEY', 'NEO4J_URI', 'NEO4J_USER', 'NEO4J_PASSWORD']:
    os.environ[k] = userdata.get(k) or ''
os.environ['NEO4J_DATABASE'] = 'neo4j'
os.environ['EMBED_DEVICE'] = 'cuda'
os.environ['PYTHONIOENCODING'] = 'utf-8'

## 3.2 Sync data từ GDrive → /content (faster IO)

GDrive IO chậm; copy hot data sang `/content` (ephemeral nhưng nhanh).

In [ ]:
GDRIVE_DATA = '/content/drive/MyDrive/legal-graph-kb-results/data/eval'
!rm -rf data/eval && mkdir -p data/eval
!rsync -a $GDRIVE_DATA/ data/eval/
!du -sh data/eval/* | sort -h

## 3.3 (Optional) Backfill plain_answer cho elite arms

Chỉ cần nếu inference cũ KHÔNG có plain_answer field. Cost ~$0.72, time ~1h trên Colab parallel.

In [ ]:
# Check xem có cần backfill không
import json
from pathlib import Path
for combo_dir in (list(Path('data/eval/results').iterdir()) +
                   list(Path('data/eval/multimodel/results').iterdir())):
    if not combo_dir.is_dir(): continue
    if not combo_dir.name.startswith('elite_'): continue
    sample = next(combo_dir.glob('A*.json'), None)
    if not sample: continue
    has_plain = bool(json.loads(sample.read_text(encoding='utf-8')).get('plain_answer'))
    print(f'  {combo_dir.name:<40} plain_answer: {"✓" if has_plain else "✗ needs backfill"}')

In [ ]:
# Run backfill (uncomment nếu cần)
# !python -m experiments.rerender_plain_answer --combos all

## 3.4 Apply audit fixes (v1 + v2)

- v1: fix pairwise _vote bug → re-aggregate pairwise from cache
- v2: split hallucination metric + tag API errors

In [ ]:
# Audit fixes — chỉ chạy nếu metrics.json đã có (after compute_metrics first pass)
import os
if os.path.exists('data/eval/metrics.json'):
    !python -m experiments.audit_apply_fixes
    !python -m experiments.audit_apply_fixes_v2
else:
    print('Run compute_metrics first, then re-run this cell.')

## 3.5 Compute metrics — R1 (5-arm)

In [ ]:
!python -m experiments.compute_metrics 2>&1 | tail -30

## 3.6 Compute metrics — R2 (multimodel)

In [ ]:
!python -m experiments.compute_multimodel_metrics 2>&1 | tail -30

## 3.7 Re-apply audit fixes sau compute (giờ có metrics.json mới)

In [ ]:
!python -m experiments.audit_apply_fixes
!python -m experiments.audit_apply_fixes_v2

## 3.8 Generate reports

In [ ]:
!python -m experiments.generate_report 2>&1 | tail -5
!python -m experiments.generate_multimodel_report 2>&1 | tail -5
!python -m experiments.compute_significance 2>&1 | tail -5

## 3.9 Display key report tables

In [ ]:
from IPython.display import Markdown, display
for f in ['reports/FINAL_REPORT.md', 'reports/experiment_report.md',
           'reports/multimodel_report.md', 'reports/significance.md']:
    if os.path.exists(f):
        print(f'\n{"="*70}\n{f}\n{"="*70}')
        with open(f, encoding='utf-8') as fh:
            display(Markdown(fh.read()[:5000]))

## 3.10 Sync results back → GDrive

In [ ]:
GDRIVE_OUT = '/content/drive/MyDrive/legal-graph-kb-results'
!mkdir -p $GDRIVE_OUT/reports $GDRIVE_OUT/data/eval
!rsync -av reports/ $GDRIVE_OUT/reports/ | tail -10
!rsync -av --include='*.json' --include='*.jsonl' --include='*.csv' --exclude='results/' --exclude='multimodel/results/' data/eval/ $GDRIVE_OUT/data/eval/ | tail -10

## 3.11 Download reports zip (for local)

In [ ]:
!cd /content/legal-graph-kb && zip -r /content/reports_bundle.zip reports/ data/eval/metrics.json data/eval/multimodel/metrics.json data/eval/metrics.csv
from google.colab import files
files.download('/content/reports_bundle.zip')

## Done

Reports + metrics đều có trong:
- `/content/legal-graph-kb/reports/` (ephemeral)
- `/content/drive/MyDrive/legal-graph-kb-results/reports/` (persistent)
- `reports_bundle.zip` đã download về local